# Chapter 2 -- The Agent Loop from Scratch (AWS Bedrock, Claude via boto3 Converse)

Work through this notebook **after reading** `notes/ch02-agent-loop-from-scratch.md`. This is a *third* variant of this chapter's agent loop, alongside the Anthropic-SDK Bedrock notebook (`ch02-agent-loop-solved.ipynb`, native `tool_use`/`tool_result` wire format) and the local Ollama notebook (`ch02-agent-loop-ollama-solved.ipynb`, OpenAI wire format). This one talks to Claude on AWS Bedrock through **`boto3`'s Converse API** -- a third, genuinely different wire shape (`stopReason`, `toolUse`, `toolResult` in camelCase) that the notes deliberately don't teach directly (see notes Section 1), but is worth knowing hands-on since it's the plain, stable way to call Bedrock without the Anthropic SDK.

Three exercises below have a stub to fill in: the **budget guard**, **tool-result batching**, and **JSONL trajectory logging** -- the same three concepts as the other two variants, expressed in Converse's own shape. Every exercise is verified directly against your real AWS Bedrock account, right below it -- there is no scripted fake client, following this repo's established convention (see `OLLAMA-NOTEBOOK-GUIDE.md`).

## One-time AWS setup

This notebook needs a real AWS account with Bedrock model access granted for at least one Claude model -- there is no free/local option for this variant (that's what the Ollama notebook is for).

1. **Credentials.** Copy `.env.example` to `.env` at the repo root (if you haven't already) and fill in `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and `AWS_REGION` for an IAM user/role with Bedrock permissions.
2. **Model access.** In the AWS Console, go to **Amazon Bedrock -> Model access** and request access to the Claude models you want to use. This notebook is set up for **Claude Sonnet 4.6** and **Claude Haiku 4.5** -- both need to show as granted before this notebook will work.
3. **Inference profile, not a bare model ID.** Current-generation Claude models on Bedrock reject on-demand invocation by their plain model ID (e.g. `anthropic.claude-sonnet-4-6`) with a `ValidationException` telling you to use an inference profile instead. The `us.` prefix below (`us.anthropic.claude-sonnet-4-6`) is that inference profile ID, found by calling `boto3.client("bedrock").list_inference_profiles()` -- if a model you add later gives the same error, look it up the same way rather than guessing at the prefix.
4. **AWS Marketplace subscription.** The first real call to a given model can fail with `AccessDeniedException: INVALID_PAYMENT_INSTRUMENT` if your account's AWS Marketplace subscription for that model hasn't fully provisioned -- this can happen even on an account with valid credits. If you hit this, check **Billing -> Payment methods** and **AWS Marketplace -> Manage subscriptions**, wait a few minutes, and retry; it is an account state issue, not a code bug.

To switch which model this notebook calls, edit `BEDROCK_MODEL` in the connection cell below -- comment out the active line, uncomment the other.

In [1]:
%pip install boto3 python-dotenv


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv
import boto3


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable this notebook.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

# Both need "Model access" granted in the Bedrock console, and both need the
# "us." cross-region inference profile prefix -- the bare model ID is rejected
# for on-demand invocation of current-generation Claude models on Bedrock.
BEDROCK_MODEL_SONNET = "us.anthropic.claude-sonnet-4-6"
BEDROCK_MODEL_HAIKU = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

BEDROCK_MODEL = BEDROCK_MODEL_SONNET
# BEDROCK_MODEL = BEDROCK_MODEL_HAIKU

client = None
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    client = boto3.client(
        "bedrock-runtime", region_name=AWS_REGION,
        aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )


def test_connection(client, model_id):
    if client is None:
        print("AWS credentials not found in .env -- skipping connection test.")
        print("Every exercise below needs this connection to run for real.")
        return
    print(f"Testing connection to AWS Bedrock (model={model_id})...")
    try:
        response = client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": "Reply with exactly the word: pong"}]}],
            inferenceConfig={"maxTokens": 10},
        )
        reply = response["output"]["message"]["content"][0]["text"]
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("Common causes: model access not granted for this model in the Bedrock")
        print("console, a missing inference profile prefix, or an AWS Marketplace")
        print("subscription/payment-method issue on the account -- see the setup cell above.")


test_connection(client, BEDROCK_MODEL)

Testing connection to AWS Bedrock (model=us.anthropic.claude-sonnet-4-6)...
  Connection check FAILED: AccessDeniedException: An error occurred (AccessDeniedException) when calling the Converse operation: Model access is denied due to INVALID_PAYMENT_INSTRUMENT:A valid payment instrument must be provided.. Your AWS Marketplace subscription for this model cannot be completed at this time. If you recently fixed this issue, try again after 2 minutes.

Common causes: model access not granted for this model in the Bedrock
console, a missing inference profile prefix, or an AWS Marketplace
subscription/payment-method issue on the account -- see the setup cell above.


## The task and its two tools

Same task as the other two variants of this chapter: read two text files and report their combined word count. Two tools are available -- `read_file(path)` and `word_count(text)` -- the exact pair from notes Section 11's dry-run.

In [ ]:
SAMPLE_DIR = Path.cwd() / "ch02_sample_files"
SAMPLE_DIR.mkdir(exist_ok=True)

(SAMPLE_DIR / "report.txt").write_text(
    "Quarterly revenue grew twelve percent driven by strong demand in the "
    "enterprise segment and continued expansion in international markets."
)
(SAMPLE_DIR / "business.txt").write_text(
    "The board approved a new capital allocation plan focused on research "
    "and disciplined cost management across every division this year."
)

print("Sample files created:")
for f in sorted(SAMPLE_DIR.iterdir()):
    print(f"  {f.name}: {len(f.read_text().split())} words")


def read_file(path: str) -> str:
    """Return the contents of a file at `path` (resolved inside the sample directory)."""
    target = SAMPLE_DIR / path
    if not target.is_file():
        raise FileNotFoundError(f"{path} not found in {SAMPLE_DIR.name}/")
    return target.read_text()


def word_count(text: str) -> str:
    """Return the number of whitespace-separated words in `text`, as a string."""
    return str(len(text.split()))


TOOL_DISPATCH = {
    "read_file": read_file,
    "word_count": word_count,
}

# Converse API tool schema shape: each tool is a `toolSpec` object, and the
# JSON Schema itself lives nested under `inputSchema.json` -- neither OpenAI's
# top-level `parameters` nor Anthropic's top-level `input_schema`.
TOOL_CONFIG = {
    "tools": [
        {
            "toolSpec": {
                "name": "read_file",
                "description": "Read the full contents of a text file by name. Call this before word_count if you need the file content.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {"path": {"type": "string", "description": "File name, e.g. 'report.txt'"}},
                        "required": ["path"],
                    }
                },
            }
        },
        {
            "toolSpec": {
                "name": "word_count",
                "description": "Count the number of whitespace-separated words in a string of text you already have.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {"text": {"type": "string", "description": "The text to count words in"}},
                        "required": ["text"],
                    }
                },
            }
        },
    ]
}

# Real answer to compare every run's final answer against -- computed once,
# directly, with no model involved.
_COMBINED_COUNT = len(read_file("report.txt").split()) + len(read_file("business.txt").split())
print(f"Real combined word count (for later comparison): {_COMBINED_COUNT}")

## Same loop, three wire formats: Anthropic vs OpenAI/Ollama vs Bedrock Converse

This chapter now has three working variants, each speaking a genuinely different
wire format for the exact same idea. The notes teach Anthropic's native Messages
API end to end (see notes Section 1 for why); the Ollama notebook covers OpenAI's
shape; this notebook adds AWS's own **Converse API** -- Bedrock's plain,
provider-agnostic call shape, independent of both the Anthropic SDK and any
OpenAI compatibility layer.

| Concept | Anthropic Messages API | OpenAI / Ollama | Bedrock Converse API |
|---|---|---|---|
| "the model wants to act" | `stop_reason == "tool_use"` | `finish_reason == "tool_calls"` | `stopReason == "tool_use"` |
| the request(s) | `tool_use` content blocks, `input` already parsed | `tool_calls[].function.arguments` as a **JSON string** | content blocks with a `toolUse` key, `input` already parsed |
| tool schema shape | top-level `input_schema` | `function.parameters` | nested `toolSpec.inputSchema.json` |
| the answer(s) | **all** results packed into ONE `user` turn | **one separate `role: "tool"` message per call** | **all** results packed into ONE `user` turn |
| matching id field | `tool_use_id` | `tool_call_id` | `toolUseId` |
| "the model is done" | `stop_reason == "end_turn"` | `finish_reason == "stop"` | `stopReason == "end_turn"` |

The row worth sitting with here is the fourth one, because it's a genuine
two-against-one split: Anthropic and Bedrock's Converse API both batch every
tool result from one turn into a single `user`-role message -- Converse didn't
invent its own convention here, it matches Anthropic's. OpenAI is the outlier,
requiring one message per call. A second detail worth noticing: Converse hands
you `input` as an **already-parsed dict**, the same as Anthropic -- OpenAI is
the one that makes you `json.loads()` a string yourself (see `execute_tool_call`
below). Three real implementations of the same loop is enough to see which
conventions are genuinely provider-specific and which ones just happen to
agree.

In [ ]:
def execute_tool_call(tool_use_block: dict) -> tuple:
    """
    Run one Converse `toolUse` block through TOOL_DISPATCH.
    Returns (content_string, is_error) -- matches notes Section 6 exactly:
    exceptions become an error tool result, never a crash.
    Note: unlike OpenAI, `input` here is already a parsed dict, not a JSON string.
    """
    name = tool_use_block["name"]
    fn = TOOL_DISPATCH.get(name)
    if fn is None:
        return f"Error: no such tool '{name}'", True
    try:
        result = fn(**tool_use_block["input"])
        return str(result), False
    except Exception as exc:
        return f"Error: {exc}", True

## Exercises -- complete `run_agent_loop`

One function, three TODOs, matching notes Sections 5, 7, and 9 -- the same
three ideas as the Ollama notebook, expressed in Converse's own shape:

1. **TODO 1 -- budget guard.** Before each request, if `cumulative_tokens` already exceeds `max_token_budget`, print a message and return early as `(None, step - 1, cumulative_tokens)`.
2. **TODO 2 -- tool-result batching.** Unlike the OpenAI notebook, Converse expects **all** of one turn's tool results packed into a **single** `role: "user"` message, as a list of `toolResult` blocks -- matching Anthropic's convention, not OpenAI's.
3. **TODO 3 -- trajectory logging.** If `log_path` is given, append one JSON line per step (every step, including the final stop) recording `step`, `stopReason`, `cumulative_tokens`, and `tool_calls_this_step`.

Each exercise is checked directly against your real AWS Bedrock account, right below it -- see the flow diagram immediately below for how the whole function fits together before you fill anything in.

## How `run_agent_loop` Flows

This is exactly notes Section 3's "send -> read the stop signal -> branch ->
execute + append -> repeat" cycle, with Bedrock Converse's field names in place
of Anthropic's native ones (see the wire-format contrast cell above). One pass
through the `for` loop is one "turn":

```
                    ┌───────────────────────────────────────────────┐
                    │                                                │
                    ▼                                                │
        ┌─────────────────────────┐                                 │
        │ TODO 1 -- budget guard  │  cumulative_tokens > budget?     │
        │ (checked BEFORE the     │──── yes ──▶ return (None, ...)   │
        │  next call is made)     │                                  │
        └────────────┬────────────┘                                  │
                      │ no                                            │
                      ▼                                                │
        ┌────────────────────────────────┐                             │
        │ client.converse(modelId=...,   │  POST to Bedrock's          │
        │   messages=..., toolConfig=...)│  Converse API                │
        └────────────┬────────────────────┘                             │
                      ▼                                                   │
        ┌─────────────────────────┐                                      │
        │ TODO 3 -- log this step │  every step, whatever stopReason      │
        │ to trajectory_bedrock   │  turns out to be                      │
        │ _anthropic.jsonl        │                                        │
        └────────────┬────────────┘                                        │
                      ▼                                                     │
        ┌─────────────────────────────────┐                                 │
        │ append response["output"]       │  full message, toolUse blocks  │
        │ ["message"] as one assistant    │  and all -- BEFORE acting       │
        │ turn to `messages`              │                                  │
        └────────────┬─────────────────────┘                                 │
           ┌──────────┴───────────┐                                          │
           ▼                      ▼                                          │
   stopReason ==             stopReason ==                                   │
     "end_turn"                "tool_use"                                    │
           │                      │                                          │
           ▼                      ▼                                          │
   return final_text    ┌──────────────────────────────┐                     │
                         │ TODO 2 -- for EVERY toolUse   │                    │
                         │ block in this turn, execute   │                   │
                         │ it, collect ONE toolResult    │                   │
                         │ block per call, then append   │                   │
                         │ ALL of them as ONE role="user"│                   │
                         │ message                       │                   │
                         └──────────────┬─────────────────┘                  │
                                        └──────────────────────────────────────┘
                                             loop back to top (step += 1)
```

Concretely, here is what a *correct* implementation's first two turns look like on
this chapter's task (idealized -- a real model won't always follow this exactly,
which is what the verification cells below are for):

```
Step 1 -- call 1:
  messages going in:  [ {"role": "user", "content": [{"text": "Read report.txt and business.txt..."}]} ]
  model responds: stopReason="tool_use", content has two toolUse blocks --
      read_file(path="report.txt"), read_file(path="business.txt")
  TODO 1 check: cumulative_tokens (0) > budget?  No -> the call above was allowed to happen.
  TODO 3 logs {"step": 1, "stopReason": "tool_use", "tool_calls_this_step": 2, ...}
  the assistant turn is appended: {"role": "assistant", "content": [...two toolUse blocks...]}
  TODO 2: both tools are executed, and BOTH toolResult blocks are packed into ONE
      role="user" message -- `messages` is now 3 entries long: user, assistant, user

Step 2 -- call 2:
  messages going in: all 3 entries above, resent in full (the API is stateless)
  model responds: stopReason="end_turn", content=[{"text": "The combined word count is ..."}]
  TODO 3 logs step 2; the assistant turn is appended; the function returns
      (final_text, 2, cumulative_tokens)
```

Notice `messages` only grows to **3** entries here, not 4 like the OpenAI
notebook's equivalent trace -- that's the direct, visible consequence of
Converse batching both tool results into one message instead of two.

In [ ]:
MAX_TOKEN_BUDGET = 5000
MAX_STEPS = 10


def run_agent_loop(client, model, tools, messages, max_steps=MAX_STEPS,
                    max_token_budget=MAX_TOKEN_BUDGET, log_path=None):
    """
    The full agent loop against Bedrock's Converse API. Requests completions,
    branches on stopReason, executes tools, guards against runaway budgets,
    and logs every step.
    Returns: (final_text_or_None, steps_taken, cumulative_tokens)
    """
    cumulative_tokens = 0
    if log_path is not None and log_path.exists():
        log_path.unlink()  # start each run with a clean trajectory file

    for step in range(1, max_steps + 1):
        # --- TODO 1: BUDGET GUARD ---
        if cumulative_tokens > max_token_budget:
            print(f"Budget exceeded ({cumulative_tokens} > {max_token_budget}). Stopping early.")
            return None, step - 1, cumulative_tokens

        response = client.converse(
            modelId=model, messages=messages, toolConfig=tools,
        )
        usage = response["usage"]
        cumulative_tokens += usage["inputTokens"] + usage["outputTokens"]

        stop_reason = response["stopReason"]
        message = response["output"]["message"]
        tool_use_blocks = [b["toolUse"] for b in message["content"] if "toolUse" in b]

        print(f"STEP {step} | stopReason={stop_reason} | "
              f"cumulative_tokens={cumulative_tokens}")

        # --- TODO 3: TRAJECTORY LOGGING (every step, regardless of stopReason) ---
        if log_path is not None:
            with open(log_path, "a") as f:
                log_record = {
                    "step": step,
                    "stopReason": stop_reason,
                    "cumulative_tokens": cumulative_tokens,
                    "tool_calls_this_step": len(tool_use_blocks),
                }
                f.write(json.dumps(log_record) + "\n")

        # The assistant's full turn must be appended BEFORE anything is executed.
        messages.append(message)

        if stop_reason == "end_turn":
            final_text = next((b["text"] for b in message["content"] if "text" in b), "")
            print(f"  Final answer: {final_text}")
            return final_text, step, cumulative_tokens

        # --- TODO 2: EXECUTE TOOLS AND APPEND ONE BATCHED RESULT MESSAGE ---
        if stop_reason == "tool_use":
            tool_result_blocks = []
            for tool_use_block in tool_use_blocks:
                content_str, is_error = execute_tool_call(tool_use_block)
                status = "ERROR" if is_error else "ok"
                print(f"    tool_use {tool_use_block['name']}({tool_use_block['input']}) -> [{status}] "
                      f"{content_str[:60]}")
                tool_result_blocks.append({
                    "toolResult": {
                        "toolUseId": tool_use_block["toolUseId"],
                        "content": [{"text": content_str}],
                        "status": "error" if is_error else "success",
                    }
                })
            messages.append({"role": "user", "content": tool_result_blocks})

    print(f"Hit max_steps ({max_steps}) without a natural stop.")
    return None, max_steps, cumulative_tokens

**Verification -- Exercise 1 (budget guard)**

Run with an *impossible* budget (`-1` -- nothing can ever be lower than the 0
tokens spent before the very first call) so the pre-flight check is guaranteed
to trip before any real API call is made. This is fully deterministic even
against a real, non-deterministic model, because it never actually reaches
AWS -- no cost, no dependency on model behavior.

In [ ]:
fresh_messages = [{"role": "user", "content": [{"text": "Read report.txt and business.txt, then tell me the combined word count."}]}]
final_text, steps_taken, tokens = run_agent_loop(
    client, model=BEDROCK_MODEL, tools=TOOL_CONFIG,
    messages=fresh_messages, max_token_budget=-1,  # impossible to satisfy -- guaranteed instant trip
)

assert final_text is None, "Expected the budget guard to stop the loop before any call was made."
assert steps_taken == 0, f"Expected zero steps taken (guard fires before step 1 even starts), got {steps_taken}."
assert tokens == 0, f"Expected zero tokens spent (no API call should have happened), got {tokens}."
print(f"PASS -- budget guard stopped the loop before any real API call was made ({tokens} tokens spent).")

**Verification -- Exercise 2 (tool-result batching)**

Run the real task against your real Bedrock model and check a structural
invariant that must hold no matter what the model actually decided to call:
every `tool_use` turn's `toolUse` blocks must all be answered by exactly one
`role: "user"` message containing exactly that many `toolResult` blocks, each
carrying the matching `toolUseId`. Unlike the OpenAI notebook, this checks for
ONE combined message, not one message per call.

In [ ]:
fresh_messages = [{"role": "user", "content": [{"text": "Read report.txt and business.txt, then tell me the combined word count. Use the tools -- do not guess."}]}]
try:
    final_text, steps_taken, tokens = run_agent_loop(
        client, model=BEDROCK_MODEL, tools=TOOL_CONFIG,
        messages=fresh_messages, max_token_budget=20_000,
    )
except Exception as exc:
    raise AssertionError(
        f"run_agent_loop raised {type(exc).__name__}: {exc} -- likely TODO 2 not filled in yet "
        "(an unresolved tool_use turn makes the next request invalid)."
    ) from exc

tool_turns = [
    i for i, m in enumerate(fresh_messages)
    if m.get("role") == "assistant" and any("toolUse" in b for b in m.get("content", []))
]
if not tool_turns:
    print("NOTE: the real model didn't call any tools this run (it guessed instead of using them).")
    print("Nothing to verify against -- just re-run this cell; models occasionally skip tools.")
else:
    for i in tool_turns:
        expected_ids = [b["toolUse"]["toolUseId"] for b in fresh_messages[i]["content"] if "toolUse" in b]
        result_message = fresh_messages[i + 1]
        assert result_message.get("role") == "user", f"Expected a role='user' message right after turn {i}."
        result_blocks = [b["toolResult"] for b in result_message["content"] if "toolResult" in b]
        assert len(result_blocks) == len(expected_ids), (
            f"Turn {i} made {len(expected_ids)} tool call(s) but the following message had "
            f"{len(result_blocks)} toolResult block(s) -- check TODO 2."
        )
        actual_ids = [b["toolUseId"] for b in result_blocks]
        assert actual_ids == expected_ids, f"toolUseId order/mismatch: expected {expected_ids}, got {actual_ids}."
    print("PASS -- every tool call the real model made got exactly one matching toolResult, batched correctly.")

print(f"Real run: {steps_taken} step(s), {tokens} cumulative tokens.")
print(f"Real combined word count for comparison: {_COMBINED_COUNT}; model said: {final_text!r}")

**Verification -- Exercise 3 (trajectory logging)**

Run once more against the real model with `log_path` set, and confirm every
step got written to disk as valid JSON -- this is checkable independent of
what the model actually said, since it only depends on TODO 3 firing once per
step.

In [ ]:
TRAJECTORY_PATH = Path.cwd() / "trajectory_bedrock_anthropic.jsonl"

fresh_messages = [{"role": "user", "content": [{"text": "Read report.txt and business.txt, then tell me the combined word count."}]}]
try:
    final_text, steps_taken, tokens = run_agent_loop(
        client, model=BEDROCK_MODEL, tools=TOOL_CONFIG,
        messages=fresh_messages, max_token_budget=20_000, log_path=TRAJECTORY_PATH,
    )
except Exception as exc:
    raise AssertionError(
        f"run_agent_loop raised {type(exc).__name__}: {exc} -- likely TODO 2 not filled in yet."
    ) from exc

assert TRAJECTORY_PATH.exists(), "No trajectory_bedrock_anthropic.jsonl was written -- check TODO 3."
lines = TRAJECTORY_PATH.read_text().strip().splitlines()
assert len(lines) == steps_taken, f"Expected {steps_taken} logged steps, found {len(lines)}."

for line in lines:
    record = json.loads(line)  # must parse as valid JSON
    assert {"step", "stopReason", "cumulative_tokens", "tool_calls_this_step"} <= record.keys(), (
        f"Missing expected keys in logged record: {record}"
    )

print(f"PASS -- {len(lines)} step(s) logged to {TRAJECTORY_PATH.name}, each record well-formed.")
print(f"Real combined word count for comparison: {_COMBINED_COUNT}; model said: {final_text!r}")
for line in lines:
    print(" ", line)

## Key Takeaways

You now have a real, self-correcting, budget-guarded, self-logging agent loop --
built from a request/response cycle, a `stopReason` branch, and a dispatch
table using **AWS Bedrock's Converse API** against a real Claude model. Compare
this notebook's `run_agent_loop` side by side with the Ollama notebook's: same
shape, same three TODOs, genuinely different wire format for every single field
name -- and, per the contrast cell above, one convention (batching tool results)
that Converse and Anthropic actually agree on, against OpenAI's different one.